# Web scraping with BeautifulSoup



### Import BeautifulSoup

First off, you will need to import the BeautifulSoup library. BS is not part of the Python standard library (i.e. it needs to be installed separately).

In [1]:
# import beautiful soup library
from bs4 import BeautifulSoup

To work with BeautifulSoup, you first require some HTML. HTML can either be loaded from a locally stored file, or it can be \`requested' from a web server over HTTP.
To use the second approach, we will utilise another Python library called `requests`, which is able to make and handle HTTP requests and responses. 

In [2]:
# import requests library
import requests

We can use the `get` method in the requests library to retrieve an HTTP response object. An HTTP request contains header fields which may give the server some additional information about the request. One of the fields is called, \`user-agent', and it tells the server what software is making the request on behalf of the user. It may be a good idea to set this header, to try to \`fool' the server into believing the request is coming via a browser.

The response object has a property, `text`, which contains the HTML that was sent in the response.

In the following example, the HTML for a web page displaying details about a product on the Tesco website is retrieved.

In [7]:
# set a user-agent to be sent with request
#headers = {
#    "user-agent":"Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/61.0.3163.100 Safari/537.36"
#}
# request a resource from a specific URL. You might want to change this for your chosen website.
r  = requests.get("https://en.wikipedia.org/wiki/Premier_League")#,headers)

# put the text that is returned in the response in a variable
data = r.text

# look...some HTML has been sent in the response!
# data

The raw HTML is not very easy to work with, because it is in a semantic markup format. We need to \`parse' the HTML (i.e. split it into its component parts), which will make working with it much easier. For that we will create an object which is an instance of the BeautifulSoup class. The object will be a special kind of data structure. It will contain the HTML, but in a format we can work with.

In [6]:
# parse the raw HTML into a `soup' object
soup = BeautifulSoup(data, "html.parser")
# soup

Now that we have parsed the HTML, we can call methods of the BeautifulSoup class to access specific elements in the data.

### Extract a single element by tag name
For example, the `find` method will return the first available element with a specified tag name:

In [5]:
h1 = soup.find("h1")
h1


<h1 class="firstHeading mw-first-heading" id="firstHeading">Premier League</h1>

In [6]:
# That's a bit noisy. Let's clean.
print(h1.text)

Premier League


### Filter hyperlinks example (a basic web spider interpretation.)
Perhaps we want to build a web spider that goes from page to page. Let's start with this page as it has lots of hyperlinks.

We could grab all links like this

In [20]:
# extract all links
links = soup.find_all('a')
# let's check the first two links to make sure it worked
links[0:2]


<a href="/wiki/Wikipedia:Featured_articles" title="This is a featured article. Click here for more information."><img alt="Featured article" data-file-height="438" data-file-width="462" decoding="async" height="19" src="//upload.wikimedia.org/wikipedia/en/thumb/e/e7/Cscr-featured.svg/20px-Cscr-featured.svg.png" srcset="//upload.wikimedia.org/wikipedia/en/thumb/e/e7/Cscr-featured.svg/30px-Cscr-featured.svg.png 1.5x, //upload.wikimedia.org/wikipedia/en/thumb/e/e7/Cscr-featured.svg/40px-Cscr-featured.svg.png 2x" width="20"/></a>

In [32]:
# First few links are navigating around the page. Let's try to find something interesting
links[10]

<a href="/wiki/EFL_Championship" title="EFL Championship">EFL Championship</a>

In [54]:
# Now let's do something more complicated. Let's combine regular expressions with beautiful soup 
# Let's find the title of all elements about Sir Alex

import re
links = soup.find_all(href=re.compile("Ferguson"))
for link in links:
    title = link.get('title')
    print(title)
# Five appearances for the greatest manager of all time!

Alex Ferguson
None
Enlarge
Alex Ferguson
Alex Ferguson


### Extract a specific table example
Let's get the first table on the page.

In [10]:
table = soup.find("table",{"class":"wikitable sortable"})
# Let's not print the whole thing as it's a big table with lots of content, including styling.

### Quick sanity check
Let's make sure we have the right table. Let's have a look at the first (zeroth) row.

In [56]:
rows = table.find_all("tr")
rows[0]

<tr>
<th>Season</th>
<th>Champions</th>
<th>Runners-Up
</th></tr>

In [57]:
# Headings look good? Okay!
# Who were the most successful teams in the first year of the PL?
early_prem = rows[1]

# Row 1 is the 92-93 season, let's get the text from that row
teams = early_prem.text
teams

'\n1992–93\nManchester United\nAston Villa\n'

In [58]:
# Man Utd and Villa! 
# We have some newlines in there. Let's format it properly.
# We could go straight to dataframe from here, but let's use a list
early_prem = teams.splitlines()
early_prem

['', '1992–93', 'Manchester United', 'Aston Villa']

In [59]:
''' 
Frustrating! We still have some invalid values in there. 
Let's get the last two
We don't need to set an end value. Slicing to the rescue!
'''
teams = early_prem[2:]
teams

['Manchester United', 'Aston Villa']

In [12]:
# Lets formulate an appearance count for Villa and see how many times they appear in the table
# We will call appearances 'wins' to ease our variable naming

villa_wins = 0
for td in soup.find_all("td",text="Aston Villa"):
    villa_wins=villa_wins+1
    print(td.text)
print(villa_wins)

Aston Villa
1


In [60]:
# What about United? How many times do they appear?
utd_wins=0
for td in soup.find_all("td",text="Manchester United"):
    utd_wins=utd_wins+1
print(utd_wins)
# Wow! What a team! 9 appearances in eight years! 

#...Wait, that can't be right!

9


In [14]:
#Let's check my math
last_year = 2000
first_year=1992
print(last_year - first_year)
# This is why we check our data for sanity!
utd_wins=8;

8


In [61]:
# It looks like Arsenal are the 2nd most successful side. 
# Villa had one good season, but don't appear again. Let's try Arsenal and keep a running count again.
arsenal_wins = 0
for td in soup.find_all("td",text="Arsenal"):
    arsenal_wins = arsenal_wins+1
print(arsenal_wins)

3


In [62]:
import pandas as pd
data = [['Man Utd', utd_wins], ['Arsenal', arsenal_wins], ['Villa', villa_wins]] 
df = pd.DataFrame(data, columns = ['Team', 'Appearances in top two'])
df

,Team,Appearances in top two
0,Man Utd,9
1,Arsenal,3
2,Villa,1
